# 6. Deployment: Servis Inferensi FastAPI & Containerization

Tahap **Deployment** menyediakan endpoint API waktu-nyata untuk melayani prediksi kerapian citra alas kaki menggunakan model Machine Learning yang telah dilatih:
1. **Microservice REST API (FastAPI):** Menyediakan endpoint `/api/v1/predict` (POST) dan `/health` (GET).
2. **Dashboard Web Interaktif:** Antarmuka HTML5 bawaan di route `/` untuk pengujian langsung oleh pengguna melalui browser.
3. **Pengujian Inferensi Langsung:** Menguji fungsi inferensi model pada sampel citra aktual.


In [ ]:
import sys
from pathlib import Path
from PIL import Image


def find_project_dir():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / Path("src/1-project/proyek_sain_data/src/10-project_citra")):
            if (candidate / "3-data_preparation").is_dir() and (candidate / "6-deployment").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Folder proyek 10-project_citra tidak ditemukan dari direktori kerja.")


PROJECT_DIR = find_project_dir()
DATASET_DIR = PROJECT_DIR.parents[1] / "data"
DEPLOYMENT_DIR = PROJECT_DIR / "6-deployment"
sys.path.insert(0, str(DEPLOYMENT_DIR))
from app import extract_features_from_image, model, scaler

print("Model Loaded:", model is not None)
print("Scaler Loaded:", scaler is not None)
if model is None or scaler is None:
    raise RuntimeError("Model/scaler deployment belum tersedia. Jalankan run_pipeline.py terlebih dahulu.")

for folder_name, label in (("RAPI", "Rapi"), ("BERANTAKAN", "Berantakan")):
    candidates = sorted(
        path for path in (DATASET_DIR / folder_name).rglob("*")
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    if not candidates:
        raise FileNotFoundError(f"Tidak ada citra di folder {folder_name}.")
    sample_path = candidates[0]
    with Image.open(sample_path) as image:
        features = extract_features_from_image(image.convert("RGB"))
    scaled_features = scaler.transform([features])
    prediction = int(model.predict(scaled_features)[0])
    class_probabilities = {
        int(class_label): float(probability)
        for class_label, probability in zip(model.classes_, model.predict_proba(scaled_features)[0])
    }
    prediction_text = "Rapi" if prediction == 1 else "Berantakan"
    print(f"{sample_path.name}: aktual={label}, prediksi={prediction_text}, probabilitas_rapi={class_probabilities[1]:.3f}")